In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import timm
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from collections import Counter

# --- 1. DEVICE CONFIGURATION ---
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available(): 
        return torch.device("cuda")
    else:
        return torch.device("cpu")

device = get_device()
print(f"⚙️ Device set to: {device}")

# --- 2. DATA TRANSFORMATIONS AND DATALOADERS ---
def prepare_data(base_dir, batch_size=32):
    """Loads images from directory and returns DataLoaders and dataset sizes."""
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Create Datasets
    datasets_dict = {
        'train': datasets.ImageFolder(root=f"{base_dir}/train", transform=train_transforms),
        'val': datasets.ImageFolder(root=f"{base_dir}/val", transform=test_transforms),
        'test': datasets.ImageFolder(root=f"{base_dir}/test", transform=test_transforms)
    }
    
    # Create DataLoaders (num_workers=0 recommended for Mac/MPS)
    dataloaders_dict = {
        x: DataLoader(datasets_dict[x], batch_size=batch_size, shuffle=(x == 'train'), num_workers=0)
        for x in ['train', 'val', 'test']
    }
    
    dataset_sizes = {x: len(datasets_dict[x]) for x in ['train', 'val', 'test']}
    
    print(f"📂 Data loaded from '{base_dir}'. Classes: {datasets_dict['train'].class_to_idx}")
    return dataloaders_dict, dataset_sizes, datasets_dict['train']

# --- 3. MODEL CREATION AND LOSS FUNCTIONS ---
def create_model(num_classes=2):
    """Initializes DeiT-Tiny and adapts the head for the number of classes."""
    model = timm.create_model('deit_tiny_patch16_224', pretrained=True)
    in_features = model.head.in_features
    model.head = nn.Linear(in_features, num_classes)
    return model.to(device)

def get_criterion_balanced_standard(train_dataset):
    """Calculates class weights using the balanced heuristic (Scikit-Learn style)."""
    class_counts = Counter(train_dataset.targets)
    total_samples = len(train_dataset.targets)
    num_classes = len(class_counts) 
    
    weight_fake = total_samples / (num_classes * class_counts[0])
    weight_real = total_samples / (num_classes * class_counts[1])
    
    class_weights = torch.tensor([weight_fake, weight_real], dtype=torch.float32).to(device)
    print(f"⚖️ Optimized Balanced Weights -> Fake (0): {weight_fake:.4f} | Real (1): {weight_real:.4f}")
    
    return nn.CrossEntropyLoss(weight=class_weights)

class FocalLoss(nn.Module):
    """Focal Loss implementation to address heavy class imbalance."""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha 
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss) 
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

def get_focal_criterion(train_dataset, gamma=2.0):
    """Calculates weights and returns a FocalLoss instance."""
    class_counts = Counter(train_dataset.targets)
    total_samples = len(train_dataset.targets)
    num_classes = len(class_counts)
    
    weight_fake = total_samples / (num_classes * class_counts[0])
    weight_real = total_samples / (num_classes * class_counts[1])
    class_weights = torch.tensor([weight_fake, weight_real], dtype=torch.float32).to(device)
    
    print(f"🎯 Focal Loss enabled (Gamma: {gamma}) with weights -> Fake: {weight_fake:.4f} | Real: {weight_real:.4f}")
    return FocalLoss(alpha=class_weights, gamma=gamma)

# --- 4. TRAINING LOOP WITH EARLY STOPPING ---
def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, save_path, num_epochs=15, patience=3):
    """Trains the model with Early Stopping based on Validation Loss."""
    start_time = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    epochs_no_improve = 0
    early_stop = False

    for epoch in range(num_epochs):
        if early_stop:
            break
            
        print(f'Epoch {epoch + 1}/{num_epochs} ', end='')

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.float() / dataset_sizes[phase]

            print(f'| {phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} ', end='')

            # Early Stopping logic
            if phase == 'val':
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(best_model_wts, save_path)
                    epochs_no_improve = 0 
                else:
                    epochs_no_improve += 1
                    
                if epochs_no_improve >= patience:
                    print(f'\n🛑 Early Stopping triggered! No improvement for {patience} epochs.')
                    early_stop = True

        print() 

    time_elapsed = time.time() - start_time
    print(f'\n⏱️ Training completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'🌟 Best Val Loss: {best_loss:.4f}. Model saved to: {save_path}\n')
    
    model.load_state_dict(best_model_wts)
    return model

# --- 5. EVALUATION PHASE ---
def evaluate_model(model, test_loader, experiment_name):
    """Evaluates the model and generates classification report and confusion matrix."""
    model.eval()
    all_preds, all_labels = [], []

    print(f"🔍 Starting evaluation on Test data ({experiment_name})...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    target_names = ['Fake (0)', 'Real (1)']
    
    print("\n--- CLASSIFICATION REPORT ---")
    print(classification_report(all_labels, all_preds, target_names=target_names))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
    plt.ylabel('Ground Truth', fontweight='bold')
    plt.xlabel('Prediction', fontweight='bold')
    plt.title(f'Confusion Matrix - {experiment_name}', fontsize=12, pad=15)
    plt.show()



In [ ]:
# ============================================================
# 🔄 AUTOMATED EXPERIMENT RUNNER (6 TESTS + EARLY STOPPING)
# ============================================================

scenari = [
    ("source", "standard"), ("target", "standard"),
    ("source", "balanced"), ("target", "balanced"),
    ("source", "focal"),    ("target", "focal")
]

results_log = {}

for exp_type, loss_type in scenari:
    print(f"\n" + "="*60)
    print(f"🚀 STARTING: Dataset={exp_type.upper()} | Loss={loss_type.upper()}")
    print("="*60)
    
    # 1. Path and Title setup
    DATA_DIR = f"split_{exp_type}"
    SAVE_PATH = f"best_deit_{exp_type}_{loss_type}.pth"
    TEST_TITLE = f"{exp_type.upper()} - {loss_type.upper()}"
    
    # 2. Prepare Data
    dataloaders, sizes, train_dataset = prepare_data(DATA_DIR, batch_size=32)
    
    # 3. Create Fresh Model
    model_run = create_model()
    
    # 4. Loss Selection
    if loss_type == "focal":
        criterion = get_focal_criterion(train_dataset, gamma=2.0)
    elif loss_type == "balanced":
        criterion = get_criterion_balanced_standard(train_dataset)
    else:
        criterion = torch.nn.CrossEntropyLoss().to(device)
        print("⚠️ Using Standard CrossEntropy (No Weights).")

    # 5. Optimizer
    optimizer = torch.optim.AdamW(model_run.parameters(), lr=1e-4, weight_decay=1e-4)

    # 6. Training with Early Stopping
    model_run = train_model(
        model=model_run, 
        dataloaders=dataloaders, 
        dataset_sizes=sizes, 
        criterion=criterion, 
        optimizer=optimizer, 
        save_path=SAVE_PATH, 
        num_epochs=15, 
        patience=3 
    )

    # 7. Final Test Evaluation
    model_run.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in dataloaders['test']:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_run(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Metrics collection
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', pos_label=1)
    
    # Store results
    results_log[TEST_TITLE] = {
        'accuracy': acc,
        'recall_real': rec,
        'precision_real': prec,
        'f1_score': f1
    }
    
    # Visual check in console
    evaluate_model(model_run, dataloaders['test'], TEST_TITLE)
    print(f"✅ Completed: {TEST_TITLE} -> Acc: {acc:.4f}, Recall Real: {rec:.4f}")

print("\n\n✨ ALL EXPERIMENTS FINISHED SUCCESSFULLY! ✨")

ANALISI TECNICA STANDARD:

SOURCE:

1. Il "punto debole": La Recall sui Real (1)
Guarda il valore Recall per Real (1): 0.40.

Cosa significa: Il modello ha ignorato il 60% dei volti veri, classificandoli erroneamente come deepfake.

Perché è successo: Usando la Standard CrossEntropy (senza pesi), il modello ha trovato "conveniente" concentrarsi sulla classe Fake (la maggioranza). Per minimizzare l'errore totale, ha imparato che predire "Fake" quasi sempre lo premiava, trascurando i dettagli dei volti reali.

2. La trappola dell'Overfitting
L'Early Stopping qui è stato fondamentale. Notiamo un comportamento classico:

Train Loss: scende costantemente da 0.48 a 0.26 (il modello sta "imparando a memoria" i dati di addestramento).

Val Loss: inizia a 0.52, tocca il minimo all'Epoca 2 e poi esplode a 0.83 all'Epoca 5.

Verdetto: Il modello ha smesso di imparare caratteristiche generali dei volti e ha iniziato a memorizzare i pixel specifici del dataset source. Fermarsi all'Epoca 2 è stata la scelta corretta per non peggiorare le prestazioni.

3. Precisione vs Recall
Precision Fake (0.75): Accettabile, ma non eccelsa.

Recall Fake (0.80): Il modello tende a vedere "fake" ovunque.

Accuracy (0.68): Sembra un numero discreto, ma è ingannevole (il cosiddetto Accuracy Paradox). In un dataset sbilanciato, l'accuracy non riflette la capacità del modello di distinguere le classi difficili.

TARGET:

Ecco l'analisi tecnica del confronto tra i due primi esperimenti "Standard" (senza pesi):

1. Generalizzazione vs Memorizzazione
Mentre il set Source è andato in overfitting quasi subito (Epoca 2), il set Target ha continuato a imparare in modo utile fino all'Epoca 6.

Perché? Questo suggerisce che le immagini nel set Target sono più variegate o estratte meglio. Il modello non è riuscito a "barare" memorizzando i pixel, ma è stato costretto a imparare caratteristiche reali dei volti per abbassare la Loss.

Risultato: L'Accuracy globale è passata dal 68% (Source) all'80.40% (Target).

2. Il salto della Recall sui Real (1)
Il dato più sorprendente è la Recall della classe Real:

Source - Standard: 0.40 (il modello era "cieco" sui reali).

Target - Standard: 0.62.

Analisi: Anche senza pesi, il set Target spinge il modello a riconoscere meglio i volti veri. Tuttavia, un valore di 0.62 è ancora insufficiente per un sistema di sicurezza (stai perdendo quasi il 40% dei volti reali, scambiandoli per deepfake).

3. Stabilità della Loss
Osserva l'andamento della Validation Loss:

Nel set Source, la Val Loss è esplosa dopo il minimo (da 0.52 a 0.83).

Nel set Target, è scesa in modo molto più fluido (da 0.45 a 0.36).

Verdetto: Il modello "si fida" di più dei dati Target. L'Early Stopping è intervenuto all'Epoca 9 (salvando il modello dell'Epoca 6) non appena la curva ha iniziato a risalire, confermando che abbiamo estratto il massimo possibile senza aiuti matematici (pesi).

Cosa ci dicono questi primi due test?
Abbiamo stabilito che il Target Dataset è qualitativamente superiore o più bilanciato intrinsecamente rispetto al Source per questo modello DeiT.

SOURCE-BALANCED_
Questo terzo risultato è la conferma definitiva di quanto i pesi bilanciati cambino radicalmente il comportamento del modello, anche su un dataset ostico come il SOURCE.

Ecco l'analisi tecnica del passaggio da Standard a Balanced per il dataset Source:

1. Il "Miracolo" della Recall sui Real (1)
Confrontiamo i dati del Test Set per la classe Real (1) su SOURCE:

Source - Standard: Recall 0.40

Source - Balanced: Recall 0.63

Analisi: Assegnando un peso di 1.47 alla classe Real (quasi il doppio rispetto alla classe Fake), abbiamo costretto il DeiT a prestare attenzione ai volti veri. Il modello ora "vede" il 23% di volti reali in più che prima ignorava completamente.

2. Il Trade-off: Precisione vs Recall
Noterai che l'Accuracy globale è leggermente scesa (da 0.68 a 0.66).

Perché? Questo è il classico "prezzo da pagare" del bilanciamento. Il modello è diventato molto più propenso a classificare le immagini come "Real". Questo ha alzato la Recall (meno Real persi), ma ha abbassato la Precision dei Real (0.46): significa che ora il modello scambia più spesso dei Fake per Real (Falsi Positivi).

3. L'instabilità del set SOURCE
L'addestramento conferma che il dataset Source è molto instabile per questo modello:

Val Loss: Ha toccato il minimo all'Epoca 2 (0.51) per poi schizzare a 0.81 all'Epoca 5.

Overfitting: Nonostante i pesi, il modello continua a imparare troppo velocemente i dati di training (Train Loss scende a 0.31) senza riuscire a generalizzare bene sul validation set.

Conclusione: Il set Source sembra contenere dei "bias" (forse sfondi o luci ripetitivi) che il Transformer memorizza invece di imparare a distinguere i lineamenti dei deepfake.